# 2D Transient Heat Plate — Separation of Variables in $X Y T$

We now solve the **time-dependent heat equation** on the same rectangular plate:

$$u_t = \alpha\,(u_{xx}+u_{yy}), \qquad 0<x<L,\;0<y<H,\;t>0$$

with mixed boundary conditions

- Bottom: $u(x,0,t)=0$
- Top: $u(x,H,t)=f(x)$
- Left: $u_x(0,y,t)=0$
- Right: $u_x(L,y,t)=0$

and an initial temperature field

$$u(x,y,0)=g(x,y).$$

## Why we do not separate $u=X(x)Y(y)T(t)$ directly

Because the top boundary condition is **non-homogeneous**: $u(x,H,t)=f(x)$.
The standard fix is to split

$$u(x,y,t)=u_s(x,y)+v(x,y,t),$$

where $u_s$ is the steady-state solution satisfying

$$u_{s,xx}+u_{s,yy}=0, \qquad u_s(x,0)=0, \quad u_s(x,H)=f(x), \quad u_{s,x}(0,y)=u_{s,x}(L,y)=0.$$

Then $v$ satisfies the homogeneous heat problem

$$v_t=\alpha(v_{xx}+v_{yy}),$$
$$v(x,0,t)=v(x,H,t)=0, \qquad v_x(0,y,t)=v_x(L,y,t)=0,$$

with initial condition

$$v(x,y,0)=g(x,y)-u_s(x,y).$$

Now separation of variables works cleanly with

$$v(x,y,t)=X(x)Y(y)T(t).$$

Substitute into the PDE:

$$X(x)Y(y)T'(t)=\alpha\left(X''(x)Y(y)T(t)+X(x)Y''(y)T(t)\right).$$

Divide by $\alpha X Y T$:

$$\frac{T'(t)}{\alpha T(t)}=\frac{X''(x)}{X(x)}+\frac{Y''(y)}{Y(y)}.$$

The left side depends only on $t$, while the right side depends only on $x$ and $y$, so both must equal the same constant. Write

$$\frac{T'}{\alpha T}=-\lambda.$$

Then the time equation is

$$T'(t)+\alpha\lambda T(t)=0,$$

with solution

$$T(t)=C\,e^{-\alpha\lambda t}.$$

So the decay rate is determined entirely by the spatial separation constant $\lambda$. To find the allowed values of $\lambda$, separate the spatial part once more:

$$\frac{X''}{X}+\frac{Y''}{Y}=-\lambda.$$

Rearrange as

$$\frac{X''}{X}+\lambda=-\frac{Y''}{Y}.$$

Again, the left side depends only on $x$ and the right side only on $y$, so each must equal another constant. Choose that constant to match the boundary conditions:

$$X''+\mu X=0, \qquad Y''+(\lambda-\mu)Y=0.$$

From the Neumann side-wall conditions $X'(0)=X'(L)=0$, the admissible $X$ eigenfunctions are

$$X_n(x)=\cos\!\left(\frac{n\pi x}{L}\right), \qquad \mu_n=\left(\frac{n\pi}{L}\right)^2, \qquad n=0,1,2,\dots$$

From the homogeneous Dirichlet conditions $Y(0)=Y(H)=0$, the admissible $Y$ eigenfunctions are

$$Y_m(y)=\sin\!\left(\frac{m\pi y}{H}\right), \qquad \lambda-\mu_n=\left(\frac{m\pi}{H}\right)^2, \qquad m=1,2,3,\dots$$

Therefore each separated mode has total spatial eigenvalue

$$\lambda_{nm}=\mu_n+\left(\frac{m\pi}{H}\right)^2=\left(\frac{n\pi}{L}\right)^2+\left(\frac{m\pi}{H}\right)^2.$$

That is where the exponential time factor comes from:

$$T_{nm}(t)=e^{-\alpha\lambda_{nm}t}=\exp\!\left[-\alpha\left(\left(\frac{n\pi}{L}\right)^2+\left(\frac{m\pi}{H}\right)^2\right)t\right].$$

So the transient part is

$$v(x,y,t)=\sum_{n=0}^{\infty}\sum_{m=1}^{\infty} b_{nm}\,\cos\!\left(\frac{n\pi x}{L}\right)\sin\!\left(\frac{m\pi y}{H}\right)e^{-\alpha\lambda_{nm}t},$$

where

$$\lambda_{nm}=\left(\frac{n\pi}{L}\right)^2+\left(\frac{m\pi}{H}\right)^2.$$

Hence the full solution is

$$u(x,y,t)=u_s(x,y)+v(x,y,t).$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# Physical and geometric parameters
alpha = 1.0
L = 2.0
H = 1.0

# Numerical grid
Nx, Ny = 180, 120
x = np.linspace(0.0, L, Nx)
y = np.linspace(0.0, H, Ny)
X, Y = np.meshgrid(x, y)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Number of modes
Nn_steady = 50      # cosine modes for steady state in x
Nn_transient = 24   # cosine modes in x for transient part
Nm_transient = 24   # sine modes in y for transient part

def f(x):
    """Top boundary condition u(x,H,t) = f(x)."""
    return np.sin(np.pi * x / L) ** 2

def g(x, y):
    """Initial condition u(x,y,0) = g(x,y)."""
    return np.zeros_like(x)

fx = f(x)
G = g(X, Y)

print(f"Plate dimensions: L={L}, H={H}, alpha={alpha}")
print(f"Grid: {Nx} x {Ny}")
print(f"Steady modes: {Nn_steady}, transient modes: {Nn_transient} x {Nm_transient}")

In [ ]:
# Build the steady-state solution u_s(x,y) from the cosine-series solution
a = np.zeros(Nn_steady)

integral_0, _ = quad(f, 0.0, L)
a[0] = integral_0 / (L * H)

for n in range(1, Nn_steady):
    kn = n * np.pi / L
    integrand = lambda s, n=n: f(s) * np.cos(n * np.pi * s / L)
    integral_n, _ = quad(integrand, 0.0, L)
    a[n] = (2.0 / L) * integral_n / np.sinh(kn * H)

U_steady = a[0] * Y
for n in range(1, Nn_steady):
    kn = n * np.pi / L
    U_steady += a[n] * np.sinh(kn * Y) * np.cos(kn * X)

V0 = G - U_steady

print("Steady-state and transient initial remainder computed.")
print(f"max|u_s(x,H)-f(x)| = {np.max(np.abs(U_steady[-1, :] - fx)):.2e}")
print(f"max|u_s(x,0)|      = {np.max(np.abs(U_steady[0, :])):.2e}")
print(f"Initial transient range: [{V0.min():.3f}, {V0.max():.3f}]")

In [ ]:
# Compute transient coefficients b_nm for v(x,y,0) = g(x,y) - u_s(x,y)

B = np.zeros((Nn_transient, Nm_transient + 1))

for n in range(Nn_transient):
    cos_n = np.cos(n * np.pi * X / L)
    x_weight = 1.0 if n == 0 else 2.0
    for m in range(1, Nm_transient + 1):
        sin_m = np.sin(m * np.pi * Y / H)
        integrand = V0 * cos_n * sin_m
        integral_xy = np.trapz(np.trapz(integrand, x, axis=1), y)
        B[n, m] = (x_weight / (L * H)) * 2.0 * integral_xy

def transient_solution(t):
    V = np.zeros_like(X)
    for n in range(Nn_transient):
        cos_n = np.cos(n * np.pi * X / L)
        kn2 = (n * np.pi / L) ** 2
        for m in range(1, Nm_transient + 1):
            lam_nm = kn2 + (m * np.pi / H) ** 2
            V += B[n, m] * cos_n * np.sin(m * np.pi * Y / H) * np.exp(-alpha * lam_nm * t)
    return V

times = [0.0, 0.02, 0.08, 0.25, 1.0]
solutions = [U_steady + transient_solution(t) for t in times]

print(f"Max reconstruction error at t=0: {np.max(np.abs(solutions[0] - G)):.2e}")
print(f"Max top-boundary mismatch at t=1: {np.max(np.abs(solutions[-1][-1, :] - fx)):.2e}")

In [ ]:
# Visualize the transition toward steady state
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
axes = axes.ravel()

vmin = min(U.min() for U in solutions)
vmax = max(U.max() for U in solutions)

for ax, t, U in zip(axes[:5], times, solutions):
    cf = ax.contourf(X, Y, U, levels=40, cmap="hot", vmin=vmin, vmax=vmax)
    ax.contour(X, Y, U, levels=10, colors="k", linewidths=0.3, alpha=0.4)
    ax.set_title(f"t = {t:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

axes[5].plot(x, fx, color="black", lw=2.5, label="top BC f(x)")
for t, U in zip(times, solutions):
    axes[5].plot(x, U[-1, :], lw=1.5, label=f"u(x,H,{t:.2f})")
axes[5].set_title("Top boundary check")
axes[5].set_xlabel("x")
axes[5].set_ylabel("temperature")
axes[5].grid(True, alpha=0.3)
axes[5].legend(fontsize=8)

fig.colorbar(cf, ax=axes[:5], shrink=0.85, label="u(x,y,t)")
plt.show()